<a href="https://www.kaggle.com/code/thisisrick25/imdb-text-preprocessing-bow-assignment?scriptVersionId=355141248" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# IMDb Movie Reviews — Complete Preprocessing and Bag-of-Words Pipeline

This notebook implements the complete preprocessing pipeline requested in the assignment using the **IMDb Dataset of 50K Movie Reviews**.

The specified Kaggle dataset contains 50,000 movie reviews and the CSV contains the `review` and `sentiment` columns.

## Tasks covered

- **(a)** Lowercasing and regex-based removal of HTML tags, URLs, punctuation, and irrelevant characters.
- **(b)** Tokenization and stop-word removal.
- **(c)** Lemmatization and comparison of unique-token counts before and after lemmatization.
- **(d)** Custom `extract_word_features(review)` without `CountVectorizer`.
- **(e)** Feature dictionaries for the two supplied reviews.
- **(f)** Bag-of-Words using `CountVectorizer`.
- **(g)** Vocabulary comparison and explanation of disappeared words.

## 1. Imports and Dataset Path

The Kaggle file is normally named:

```text
IMDB Dataset.csv
```

Place it in the same directory as this notebook, or change `DATA_PATH`.

In [ ]:
# If necessary:
# %pip install -q pandas numpy nltk scikit-learn

import re
import numpy as np
import pandas as pd

from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

DATA_PATH = "/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv"

## 2. Load and Inspect the Dataset

In [ ]:
import os
import pandas as pd

# Automatically search Kaggle's input directory
dataset_path = None

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower() == "imdb dataset.csv":
            dataset_path = os.path.join(root, file)
            break
    if dataset_path:
        break

if dataset_path is None:
    raise FileNotFoundError(
        "IMDB Dataset.csv was not found under /kaggle/input/. "
        "Make sure the IMDb dataset has been added to the Kaggle notebook."
    )

print("Dataset found at:")
print(dataset_path)

# Load dataset
df = pd.read_csv(dataset_path)

print("\nDataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

# Part (a): Lowercasing and Regex Cleaning

The cleaning function:

1. converts text to lowercase,
2. removes HTML tags,
3. removes URLs,
4. removes email addresses,
5. removes punctuation, numbers, and other non-alphabetic characters,
6. normalizes whitespace.

In [ ]:
def clean_text(text):
    text = str(text).lower()

    # Remove HTML tags, e.g. <br />
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove email addresses
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.\w+\b", " ", text)

    # Keep only alphabetic characters and whitespace
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["original_review"] = df["review"].astype(str)
df["cleaned_review"] = df["original_review"].apply(clean_text)

print("Original review:")
print(df.loc[0, "original_review"][:1000])

print("\nCleaned review:")
print(df.loc[0, "cleaned_review"][:1000])

### Regex cleaning demonstration

In [ ]:
regex_demo = (
    "WOW!!! <br /> This movie is amazing! "
    "Visit https://example.com or email test@example.com."
)

print("Before:")
print(regex_demo)

print("\nAfter:")
print(clean_text(regex_demo))

# Part (b): Tokenization and Stop-Word Removal

In [ ]:
import nltk

nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))

def tokenize_and_remove_stopwords(text):
    tokens = word_tokenize(text)

    return [
        token for token in tokens
        if token not in stop_words
    ]

df["tokens_before_lemmatization"] = (
    df["cleaned_review"].apply(tokenize_and_remove_stopwords)
)

print("Example tokens:")
print(df.loc[0, "tokens_before_lemmatization"][:100])

# Part (c): Lemmatization

The vocabulary is compared:

- immediately before lemmatization,
- immediately after lemmatization.

A vocabulary is the set of unique tokens appearing across all reviews.

In [ ]:
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(token) for token in tokens]

df["lemmatized_tokens"] = (
    df["tokens_before_lemmatization"].apply(lemmatize_tokens)
)

vocabulary_before_lemmatization = {
    token
    for tokens in df["tokens_before_lemmatization"]
    for token in tokens
}

vocabulary_after_lemmatization = {
    token
    for tokens in df["lemmatized_tokens"]
    for token in tokens
}

print("Unique tokens before lemmatization:",
      len(vocabulary_before_lemmatization))

print("Unique tokens after lemmatization:",
      len(vocabulary_after_lemmatization))

print("Reduction:",
      len(vocabulary_before_lemmatization)
      - len(vocabulary_after_lemmatization))

In [ ]:
removed_or_merged_by_lemmatization = (
    vocabulary_before_lemmatization
    - vocabulary_after_lemmatization
)

print("Examples of vocabulary items that disappear after lemmatization:")
print(sorted(removed_or_merged_by_lemmatization)[:100])

### Create the final preprocessed text

The lemmatized tokens are joined back into strings for the Bag-of-Words step.

In [ ]:
df["preprocessed_review"] = df["lemmatized_tokens"].apply(
    lambda tokens: " ".join(tokens)
)

print(df.loc[0, "preprocessed_review"][:1000])

# Part (d): Custom `extract_word_features(review)`

This function returns a dictionary containing the frequency of each word.

**`CountVectorizer` is deliberately not used here.**

In [ ]:
def extract_word_features(review):
    # Apply the same preprocessing pipeline
    cleaned = clean_text(review)
    tokens = word_tokenize(cleaned)

    # Stop-word removal
    tokens = [
        token for token in tokens
        if token not in stop_words
    ]

    # Lemmatization
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
    ]

    # Word-frequency dictionary
    return dict(Counter(tokens))

# Part (e): Feature Dictionaries for the Given Reviews

In [ ]:
review_1 = "This movie was absolutely amazing and entertaining"
review_2 = "The movie was boring and predictable"

features_1 = extract_word_features(review_1)
features_2 = extract_word_features(review_2)

print("Review 1:")
print(review_1)
print("\nFeature dictionary:")
print(features_1)

print("\n" + "=" * 70)

print("\nReview 2:")
print(review_2)
print("\nFeature dictionary:")
print(features_2)

# Part (f): Bag-of-Words with `CountVectorizer`

For this part, `CountVectorizer` is explicitly required.

The **fully preprocessed reviews** are used to construct the document-term matrix.

In [ ]:
vectorizer = CountVectorizer()

bow_matrix = vectorizer.fit_transform(
    df["preprocessed_review"]
)

bow_vocabulary = vectorizer.get_feature_names_out()

print("Number of documents:",
      bow_matrix.shape[0])

print("Number of unique features:",
      len(bow_vocabulary))

print("Document-term matrix shape:",
      bow_matrix.shape)

In [ ]:
print("First 100 Bag-of-Words features:")
print(bow_vocabulary[:100])

# Part (g): Vocabulary Before and After Preprocessing

To explain why vocabulary items disappear, four stages are compared:

1. **Raw tokenization**
2. **After regex cleaning**
3. **After stop-word removal**
4. **After lemmatization**

This makes it possible to attribute vocabulary changes to a particular preprocessing step.

In [ ]:
# 1. Raw vocabulary: lowercase + tokenization only
raw_token_lists = df["original_review"].apply(
    lambda text: word_tokenize(str(text).lower())
)

raw_vocabulary = {
    token
    for tokens in raw_token_lists
    for token in tokens
}

# 2. Vocabulary after regex cleaning
cleaned_vocabulary = {
    token
    for text in df["cleaned_review"]
    for token in text.split()
}

# 3. Vocabulary after stop-word removal
stopword_removed_vocabulary = {
    token
    for tokens in df["tokens_before_lemmatization"]
    for token in tokens
}

# 4. Vocabulary after lemmatization
final_vocabulary = vocabulary_after_lemmatization

vocabulary_comparison = pd.DataFrame({
    "Preprocessing stage": [
        "Raw tokenization",
        "After regex cleaning",
        "After stop-word removal",
        "After lemmatization"
    ],
    "Number of unique tokens": [
        len(raw_vocabulary),
        len(cleaned_vocabulary),
        len(stopword_removed_vocabulary),
        len(final_vocabulary)
    ]
})

display(vocabulary_comparison)

## Tokens Removed by Each Stage

The following code identifies vocabulary items that disappear between stages.

In [ ]:
removed_by_regex = raw_vocabulary - cleaned_vocabulary

removed_by_stopwords = (
    cleaned_vocabulary - stopword_removed_vocabulary
)

removed_by_lemmatization = (
    stopword_removed_vocabulary - final_vocabulary
)

print("Number removed by regex cleaning:",
      len(removed_by_regex))

print("Number removed by stop-word removal:",
      len(removed_by_stopwords))

print("Number removed/merged by lemmatization:",
      len(removed_by_lemmatization))

### Examples removed by regex cleaning

These are tokens that existed during raw tokenization but are absent after the regular-expression cleaning stage. This includes material associated with HTML, URLs, punctuation, numbers, and other non-alphabetic content.

In [ ]:
print(sorted(removed_by_regex)[:150])

### Examples removed as stop words

These tokens disappear because they are present in NLTK's English stop-word list.

Examples include common function words such as `the`, `was`, `and`, `this`, and `is`.

In [ ]:
print(sorted(removed_by_stopwords)[:150])

### Examples affected by lemmatization

Lemmatization maps different grammatical forms to common lemmas. For example, a plural form such as `movies` can be mapped to `movie`.

Thus, an item can disappear as an independent vocabulary entry while its information remains represented by another lemma.

In [ ]:
print(sorted(removed_by_lemmatization)[:150])

# Final Results

This table collects the numerical results required by the assignment.

In [ ]:
final_results = pd.DataFrame({
    "Measure": [
        "Number of documents",
        "Unique tokens before lemmatization",
        "Unique tokens after lemmatization",
        "Unique features in Bag-of-Words",
        "Bag-of-Words matrix shape"
    ],
    "Result": [
        len(df),
        len(vocabulary_before_lemmatization),
        len(vocabulary_after_lemmatization),
        len(bow_vocabulary),
        str(bow_matrix.shape)
    ]
})

display(final_results)

# Conclusion

The IMDb reviews were processed using a complete NLP pipeline.

First, reviews were converted to lowercase and regular expressions were used to remove HTML tags, URLs, email addresses, punctuation, numbers, and other irrelevant characters. The cleaned reviews were tokenized and English stop words were removed.

Lemmatization was then applied, and the unique-token counts before and after lemmatization were compared. The vocabulary comparison also identifies which tokens disappeared at each stage and explains whether the cause was regex cleaning, stop-word removal, or lemmatization.

A custom `extract_word_features(review)` function was implemented without `CountVectorizer` and applied to the two required example reviews.

Finally, `CountVectorizer` was used on the fully preprocessed reviews to construct the Bag-of-Words document-term matrix. The number of documents, unique features, and matrix shape are reported directly from the resulting matrix.

## Requirement Checklist

| Requirement | Status |
|---|---|
| (a) Lowercase | ✓ |
| (a) Regex HTML removal | ✓ |
| (a) Regex URL removal | ✓ |
| (a) Regex punctuation/irrelevant-character removal | ✓ |
| (b) Tokenization | ✓ |
| (b) Stop-word removal | ✓ |
| (c) Lemmatization | ✓ |
| (c) Unique-token comparison | ✓ |
| (d) Custom `extract_word_features()` | ✓ |
| (d) No `CountVectorizer` for custom function | ✓ |
| (e) First supplied review | ✓ |
| (e) Second supplied review | ✓ |
| (f) Bag-of-Words with `CountVectorizer` | ✓ |
| (f) Number of documents | ✓ |
| (f) Number of unique features | ✓ |
| (f) Matrix shape | ✓ |
| (g) Vocabulary comparison | ✓ |
| (g) Disappeared-word identification | ✓ |
| (g) Explanation of why words disappeared | ✓ |